# Tutorial: Unsupervised Learning

> **Type:** Tutorial. Follow along step by step. Your only job is to make it work.  
> **Time:** ~2 hours  
> **Prerequisites:** Modules 6 (Python ecosystem), 9 (EDA), 15 (dimensionality reduction)  
> **What you'll build:** K-Means customer segmentation, a K-Means vs. GMM vs. DBSCAN comparison on non-spherical clusters, Isolation Forest anomaly detection, and a practical customer segmentation use case.

---

## What We're Building

Four sections, each showing a different side of unsupervised learning:
1. **K-Means** on Iris: cluster, evaluate, visualise
2. **K-Means vs. GMM vs. DBSCAN**: watch K-Means and GMM both fail on non-convex shapes, and DBSCAN succeed
3. **Isolation Forest**: anomaly detection with injected anomalies
4. **Customer segmentation**: a realistic business scenario

## Step 0: Setup

In [ ]:
# !pip install scikit-learn matplotlib seaborn numpy pandas

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris, make_moons, make_circles, make_blobs
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score

SEED = 42
np.random.seed(SEED)
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')
print('Setup complete.')

---
## Part 1: K-Means Clustering on Iris

### Step 1: Load and Inspect

In [ ]:
iris = load_iris()
X_iris, y_iris = iris.data, iris.target  # y_iris NOT used in training
feature_names = iris.feature_names

print(f"Iris dataset: {X_iris.shape[0]} samples, {X_iris.shape[1]} features")
print(f"True classes: {iris.target_names} (we WON'T use these during clustering)")
print(f"\nFirst 5 rows:")
pd.DataFrame(X_iris, columns=feature_names).head()

### Step 2: Find the Best K — Elbow Method and Silhouette Score

In [ ]:
k_values   = range(2, 11)
inertias   = []
silhouettes = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = km.fit_predict(X_iris)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_iris, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(k_values, inertias, marker='o', linewidth=2, color='#4E79A7')
axes[0].set_xlabel('Number of clusters (K)')
axes[0].set_ylabel('Inertia (sum of squared distances)')
axes[0].set_title('Elbow Method', fontweight='bold')
axes[0].axvline(3, color='red', linestyle='--', linewidth=1.5, label='True K=3')
axes[0].legend()

axes[1].plot(k_values, silhouettes, marker='o', linewidth=2, color='#E15759')
axes[1].set_xlabel('Number of clusters (K)')
axes[1].set_ylabel('Silhouette Score (higher = better)')
axes[1].set_title('Silhouette Score', fontweight='bold')
axes[1].axvline(3, color='red', linestyle='--', linewidth=1.5, label='True K=3')
axes[1].legend()

plt.tight_layout()
plt.savefig('kmeans_k_selection.png', dpi=150, bbox_inches='tight')
plt.show()

best_k = list(k_values)[np.argmax(silhouettes)]
print(f"Best K by silhouette: {best_k}")

> **What just happened?** The elbow at K=3 matches the true number of species, which we didn't use. The silhouette score peaks at K=2, though. Two of the three species are very similar and hard to separate without labels, so the two metrics disagree. That's normal: pick K with both metrics and domain knowledge, not one number.

### Step 3: Cluster and Visualise in 2D with PCA

In [ ]:
km = KMeans(n_clusters=3, random_state=SEED, n_init=10)
cluster_labels = km.fit_predict(X_iris)

# Compare to true labels using Adjusted Rand Index
ari = adjusted_rand_score(y_iris, cluster_labels)
print(f"Adjusted Rand Index vs true labels: {ari:.3f}  (1.0 = perfect, 0 = random)")

# PCA to 2D for visualisation
pca = PCA(n_components=2, random_state=SEED)
X_2d = pca.fit_transform(X_iris)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colours = ['#4E79A7', '#E15759', '#F28E2B']

# Cluster assignments
for i, c in enumerate(colours):
    mask = cluster_labels == i
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1], color=c, s=50, 
                    alpha=0.7, label=f'Cluster {i}')
# Plot centroids in PCA space
centroids_2d = pca.transform(km.cluster_centers_)
axes[0].scatter(centroids_2d[:, 0], centroids_2d[:, 1], 
                marker='X', s=200, color='black', zorder=5, label='Centroids')
axes[0].set_title('K-Means Cluster Assignments (PCA)', fontweight='bold')
axes[0].legend()
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')

# True species
for i, (name, c) in enumerate(zip(iris.target_names, colours)):
    mask = y_iris == i
    axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1], color=c, s=50, alpha=0.7, label=name)
axes[1].set_title('True Species Labels (PCA)', fontweight='bold')
axes[1].legend()
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')

plt.tight_layout()
plt.savefig('kmeans_iris.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 2: K-Means vs GMM vs DBSCAN on Non-Spherical Clusters

### Step 4: Generate Challenging Data

In [ ]:
# Three challenging datasets where K-Means struggles
X_moons, y_moons = make_moons(n_samples=300, noise=0.1, random_state=SEED)
X_circles, y_circles = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=SEED)
X_blobs, y_blobs = make_blobs(n_samples=300, centers=3, cluster_std=[0.5, 2.5, 0.5], random_state=SEED)

datasets = [
    (X_moons,   'Two Moons'),
    (X_circles, 'Concentric Circles'),
    (X_blobs,   'Unequal Variance Blobs'),
]

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 14))
palette = ['#4E79A7', '#E15759', '#F28E2B', '#59A14F']

for col, (X, title) in enumerate(datasets):
    # K-Means (row 0)
    km = KMeans(n_clusters=2, random_state=SEED, n_init=10)
    km_labels = km.fit_predict(X)
    
    for lbl in np.unique(km_labels):
        mask = km_labels == lbl
        axes[0, col].scatter(X[mask, 0], X[mask, 1], 
                              color=palette[lbl], s=20, alpha=0.7)
    axes[0, col].set_title(f'{title}\nK-Means', fontweight='bold')

    # GMM (row 1): elliptical covariance, still assumes convex clusters
    gm = GaussianMixture(n_components=2, random_state=SEED)
    gm_labels = gm.fit_predict(X)

    for lbl in np.unique(gm_labels):
        mask = gm_labels == lbl
        axes[1, col].scatter(X[mask, 0], X[mask, 1],
                              color=palette[lbl], s=20, alpha=0.7)
    axes[1, col].set_title(f'{title}\nGMM', fontweight='bold')
    
    # DBSCAN (row 2)
    db = DBSCAN(eps=0.3, min_samples=5)
    db_labels = db.fit_predict(X)
    n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
    n_noise    = (db_labels == -1).sum()
    
    for lbl in np.unique(db_labels):
        mask = db_labels == lbl
        colour = 'grey' if lbl == -1 else palette[lbl % len(palette)]
        size   = 10 if lbl == -1 else 20
        label  = f'Noise ({n_noise})' if lbl == -1 else f'Cluster {lbl}'
        axes[2, col].scatter(X[mask, 0], X[mask, 1], 
                              color=colour, s=size, alpha=0.7, label=label)
    axes[2, col].set_title(f'{title}\nDBSCAN ({n_clusters} clusters, {n_noise} noise)',
                            fontweight='bold')
    axes[2, col].legend(fontsize=7)

axes[0, 0].set_ylabel('K-Means', fontsize=12)
axes[1, 0].set_ylabel('GMM', fontsize=12)
axes[2, 0].set_ylabel('DBSCAN', fontsize=12)
plt.suptitle('K-Means vs GMM vs DBSCAN on Non-Spherical Clusters', 
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('kmeans_vs_dbscan.png', dpi=150, bbox_inches='tight')
plt.show()

> **What just happened?** K-Means assumes spherical, evenly sized clusters, so it can't recover the moons or the circles. GMM relaxes that to elliptical clusters, but still treats each cluster as a single convex blob. On the moons it does no better than K-Means (Part 5 scores both at an ARI of 0.479): stretching and rotating an ellipse still can't wrap around a crescent. DBSCAN finds the correct shapes, because it groups points by density rather than fitting a distribution, so it doesn't make that convexity assumption at all.

---
## Part 3: Anomaly Detection with Isolation Forest

### Step 5: Generate Normal Data with Injected Anomalies

In [ ]:
# Normal data: Gaussian
n_normal   = 400
n_anomalies = 20
X_normal   = np.random.randn(n_normal, 2)

# Anomalies scattered far from the normal cloud
X_anomalies = np.random.uniform(low=-4, high=4, size=(n_anomalies, 2))
X_anomalies = X_anomalies[np.abs(X_anomalies).sum(axis=1) > 5][:n_anomalies]
# Fall back if not enough far points
if len(X_anomalies) < n_anomalies:
    X_anomalies = np.column_stack([
        np.random.choice([-4, 4], n_anomalies) + np.random.randn(n_anomalies) * 0.3,
        np.random.uniform(-4, 4, n_anomalies)
    ])

X_all = np.vstack([X_normal, X_anomalies])
y_true = np.hstack([np.ones(n_normal), -np.ones(len(X_anomalies))])

print(f"Normal samples: {n_normal}")
print(f"Anomaly samples: {len(X_anomalies)}")
print(f"Contamination rate: {len(X_anomalies)/len(X_all):.2%}")

In [ ]:
# Train Isolation Forest
contamination = len(X_anomalies) / len(X_all)
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=contamination,  # expected fraction of anomalies
    random_state=SEED
)
pred_labels  = iso_forest.fit_predict(X_all)   # 1 = normal, -1 = anomaly
anomaly_score = iso_forest.score_samples(X_all)  # lower = more anomalous

# Evaluation
from sklearn.metrics import classification_report
print("Isolation Forest Performance:")
print(classification_report(y_true, pred_labels, target_names=['Anomaly', 'Normal']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# What the model found
colours_pred = np.where(pred_labels == 1, '#4E79A7', '#E15759')
sizes_pred   = np.where(pred_labels == 1, 20, 80)
axes[0].scatter(X_all[:, 0], X_all[:, 1], c=colours_pred, s=sizes_pred, alpha=0.7)
axes[0].set_title('Isolation Forest Predictions\n(blue=normal, red=detected anomaly)', fontweight='bold')

# Anomaly score heatmap
xx, yy = np.meshgrid(np.linspace(-5, 5, 100), np.linspace(-5, 5, 100))
Z = iso_forest.score_samples(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

cs = axes[1].contourf(xx, yy, Z, levels=20, cmap='RdYlGn')
plt.colorbar(cs, ax=axes[1], label='Anomaly score (lower = more anomalous)')
axes[1].scatter(X_all[:, 0], X_all[:, 1], c=colours_pred, s=sizes_pred * 0.5, alpha=0.5)
axes[1].set_title('Anomaly Score Landscape', fontweight='bold')

plt.tight_layout()
plt.savefig('isolation_forest.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 4: Customer Segmentation — Business Use Case

### Step 6: Synthetic E-commerce Customer Data

In [ ]:
np.random.seed(SEED)
n_customers = 500

# Generate synthetic customers with 4 natural segments
segments = {
    'High Value Loyalists':    (0.15, {'spend': (500, 100), 'frequency': (20, 3),  'recency': (5, 3)}),
    'Occasional Big Spenders': (0.20, {'spend': (400, 150), 'frequency': (4, 1),   'recency': (30, 20)}),
    'Frequent Low Spenders':   (0.35, {'spend': (50, 20),   'frequency': (25, 5),  'recency': (7, 5)}),
    'Churned':                 (0.30, {'spend': (80, 60),   'frequency': (2, 1),   'recency': (120, 40)}),
}

records = []
for seg_name, (proportion, params) in segments.items():
    n = int(n_customers * proportion)
    for _ in range(n):
        records.append({
            'avg_spend':       max(5, np.random.normal(*params['spend'])),
            'order_frequency': max(1, np.random.normal(*params['frequency'])),
            'days_since_last': max(1, np.random.normal(*params['recency'])),
            'true_segment':    seg_name
        })

df_customers = pd.DataFrame(records).sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"Customer dataset: {df_customers.shape}")
df_customers.describe().round(1)

In [ ]:
# Cluster on RFM features (Recency, Frequency, Monetary)
features = ['avg_spend', 'order_frequency', 'days_since_last']
X_rfm = df_customers[features].values

scaler_rfm = StandardScaler()
X_rfm_s = scaler_rfm.fit_transform(X_rfm)

km_rfm = KMeans(n_clusters=4, random_state=SEED, n_init=20)
df_customers['cluster'] = km_rfm.fit_predict(X_rfm_s)

# Cluster profiles
profile = df_customers.groupby('cluster')[features].mean().round(1)
profile['size'] = df_customers['cluster'].value_counts().sort_index()
print("\nCluster Profiles:")
print(profile)

In [ ]:
# Visualise clusters in 2D (PCA)
X_rfm_2d = PCA(n_components=2, random_state=SEED).fit_transform(X_rfm_s)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colours = ['#4E79A7', '#E15759', '#F28E2B', '#59A14F']

# K-Means clusters
for c in range(4):
    mask = df_customers['cluster'] == c
    axes[0].scatter(X_rfm_2d[mask, 0], X_rfm_2d[mask, 1],
                    color=colours[c], s=30, alpha=0.6, label=f'Cluster {c}')
axes[0].set_title('Customer Segments (K-Means, PCA view)', fontweight='bold')
axes[0].legend(); axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')

# Cluster profile heatmap
profile_normalised = (profile[features] - profile[features].mean()) / profile[features].std()
sns.heatmap(profile_normalised, annot=profile[features], fmt='.0f',
            cmap='RdYlGn', ax=axes[1], cbar_kws={'label': 'Z-score'})
axes[1].set_title('Cluster Profiles\n(colour=relative z-score, value=actual mean)', fontweight='bold')

plt.tight_layout()
plt.savefig('customer_segments.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n=> Label the clusters based on their profiles:")
print("   High spend + high frequency + low recency → 'High Value Loyalists'")
print("   Low recency (high days_since_last) + low frequency → 'Churned'")

---
## Part 5: Reproducing the Book's Worked Example

### Step 7: ARI Scores for K-Means, GMM, and DBSCAN on the Moons Data

Part 2 visualised K-Means, GMM, and DBSCAN on the moons data without scoring them. The book's Worked Example scores all three against the true crescent labels, using the Adjusted Rand Index.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score

X_moons_s = StandardScaler().fit_transform(X_moons)

km_moons = KMeans(n_clusters=2, random_state=SEED, n_init=10).fit(X_moons_s)
gm_moons = GaussianMixture(n_components=2, random_state=SEED).fit(X_moons_s)
db_moons = DBSCAN(eps=0.3, min_samples=5).fit(X_moons_s)

ari_km = adjusted_rand_score(y_moons, km_moons.predict(X_moons_s))
ari_gm = adjusted_rand_score(y_moons, gm_moons.predict(X_moons_s))
ari_db = adjusted_rand_score(y_moons, db_moons.labels_)
n_noise_db = int((db_moons.labels_ == -1).sum())

print(f"K-Means ARI: {ari_km:.3f}")
print(f"GMM ARI:     {ari_gm:.3f}")
print(f"DBSCAN ARI:  {ari_db:.3f}  ({n_noise_db} points labelled noise)")

### Step 8: Isolation Forest at the Book's Exact Anomaly-Detection Scale

500 points from a standard 2D normal distribution, plus 25 outliers scattered uniformly across a much wider range.

In [ ]:
rng_q = np.random.RandomState(SEED)
X_norm_wex = rng_q.normal(0, 1, size=(500, 2))
X_out_wex = rng_q.uniform(-8, 8, size=(25, 2))
X_all_wex = np.vstack([X_norm_wex, X_out_wex])
y_true_wex = np.hstack([np.ones(500), -np.ones(25)])  # 1 = normal, -1 = anomaly

iso_wex = IsolationForest(n_estimators=100, contamination=0.05, random_state=SEED)
pred_wex = iso_wex.fit_predict(X_all_wex)

n_flagged = int((pred_wex == -1).sum())
precision_wex = precision_score(y_true_wex, pred_wex, pos_label=-1)
recall_wex = recall_score(y_true_wex, pred_wex, pos_label=-1)
n_caught = int(((pred_wex == -1) & (y_true_wex == -1)).sum())

print(f"Flagged as anomalies: {n_flagged}")
print(f"Precision: {precision_wex:.3f}")
print(f"Recall: {recall_wex:.3f}")
print(f"Caught {n_caught} of 25 real anomalies")

---
## What You Built

Four working unsupervised learning pipelines: K-Means with elbow and silhouette evaluation, a K-Means vs. GMM vs. DBSCAN comparison on non-spherical data (with a clear view of where K-Means and GMM fail and DBSCAN succeeds), Isolation Forest anomaly detection with score visualisation, and a realistic customer segmentation use case with a heatmap cluster profile.

## What's Next

- Read `explanation.md` for the geometric intuitions behind K-Means, GMM, DBSCAN, and Isolation Forest.
- Use `howto.md` for the DBSCAN parameter selection guide and anomaly detection recipes.
- Keep `reference.md` open. You'll come back to its algorithm comparison table often.
- **Extension challenge:** use `UMAP` (install with `pip install umap-learn`) instead of PCA to visualise the Iris clusters. Does UMAP show the cluster structure more clearly than PCA?

---

**Next module:** `20_time_series`, where the order of observations matters and leaking future data into the past is the cardinal sin.